# Лабораторная 2: Эмбеддинг текста и визуализация

Используем офлайн-доступный пайплайн для текстовых данных: фиксированный `HashingVectorizer` (предобученная, не требующая обучения хеш-функция) + `TfidfTransformer` для взвешивания, затем понижение размерности t-SNE и визуализация. Все данные примерные и встроены в ноутбук, поэтому запуск возможен без сети.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.pipeline import make_pipeline
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Небольшой игрушечный корпус: три тематики по 5 предложений
texts = [
    "Новая модель нейросети улучшила качество машинного перевода.",
    "Исследователи представили алгоритм для ускорения обучения.",
    "Компания выпустила библиотеку с открытым исходным кодом.",
    "Разработчики обсуждают выбор оптимизатора для большой модели.",
    "Выходит обновление фреймворка для глубокого обучения.",
    "Команда выиграла финал чемпионата по футболу со счетом два ноль.",
    "Новый тренер изменил тактику и усилил защиту.",
    "Болельщики обсуждают трансфер известного нападающего.",
    "Сборная готовится к товарищескому матчу на выезде.",
    "Клуб объявил о строительстве современного стадиона.",
    "Правительство представило программу поддержки малого бизнеса.",
    "Депутаты обсуждают изменения в налоговом законодательстве.",
    "Принят новый закон о развитии цифровой экономики.",
    "Министерство объявило о реформе системы образования.",
    "Мэр города рассказал о планах по развитию общественного транспорта.",
]
labels = ["tech"] * 5 + ["sport"] * 5 + ["gov"] * 5
unique_labels = sorted(set(labels))
label_to_idx = {l: i for i, l in enumerate(unique_labels)}
y = np.array([label_to_idx[l] for l in labels])

# Фиксированный хеширующий векторизатор: не требует обучения и уже задает отображение в 512-мерное пространство
vectorizer = HashingVectorizer(n_features=512, alternate_sign=False, norm=None)
pipeline = make_pipeline(vectorizer, TfidfTransformer())
X = pipeline.fit_transform(texts)  # fit_transform только считает IDF, хеши фиксированы

print(f'Число текстов: {len(texts)}')
print(f'Размерность эмбеддинга: {X.shape[1]}')
print('Фрагмент вектора первого текста (10 значений):', X[0].toarray()[0][:10])


In [ ]:
tsne = TSNE(n_components=2, perplexity=5, learning_rate=100, random_state=42, init='random')
X_2d = tsne.fit_transform(X.toarray())

colors = {'tech': '#2E86AB', 'sport': '#E4572E', 'gov': '#76B041'}
plt.figure(figsize=(8, 6))
for label in unique_labels:
    mask = [l == label for l in labels]
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], label=label, color=colors[label], s=70, alpha=0.8)

plt.title('t-SNE визуализация эмбеддингов (Hashing + TF-IDF)')
plt.legend()
plt.xlabel('component 1')
plt.ylabel('component 2')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()
